In [2]:
import os
import sys
import django

from pathlib import Path
sys.path.append(str(Path().resolve().parent))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'support.settings')

# Add this line
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()

In [42]:
from typing import Literal

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.messages import SystemMessage, AIMessage

from staff.prompt import INTENT_PROMPT, INFO_PROMPT
from staff.schemas import IntentSchema
from author.models import Ticket, Book
from staff.knowledge import documents

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
checkpointer = InMemorySaver()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# Define the tools
def get_royality_earned(id):
    """Get the royality earning for a book."""
    book = Book.objects.get(id=id)
    return book.royality_earned

def get_royality_paid(id):
    """Get the royality pending for a book.."""
    book = Book.objects.get(id=id)
    return book.royality_earned

def get_royality_pending(id):
    """Get the royality pending for a book."""
    book = Book.objects.get(id=id)
    return book.royality_pending

def get_book_live_status(id):
    """Get a book current live status
    """
    from datetime import date
    book = Book.objects.get(id=id)
    return f"Already published on {book.pub_date}" if book.pub_date < date.today() else f"Not published yet, publication date: {book.pub_date}"


tools = [get_royality_earned, get_royality_paid,
         get_royality_pending, get_book_live_status]
# Bind the tools
llm_with_tools = llm.bind_tools(tools)

# Define the graph state
class State(MessagesState):
    book: int


def similarity_search(query):
    # get the info chunks
    chunks = documents

    # Create in-memory vector store (FAISS)
    vector_store = FAISS.from_documents(chunks, embeddings)

    # perform similarity search with score
    results = vector_store.similarity_search_with_score(query, k=1)

    # unpack result
    doc, dist = results[0]
    context = doc.page_content

    return {
        "context": context,
        "distance": float(dist)
    }


# Define Nodes
def start_graph(state: State) -> State:
    return state


def get_user_intent(state: State) -> Literal["info", "query", "complaint"]:
    prompt = INTENT_PROMPT
    prompt += f"""/n Query: {state["messages"][0].content}"""
    structured_llm = llm.with_structured_output(IntentSchema)
    response = structured_llm.invoke(prompt)
    return response.intent


def assistant(state: State) -> State:
    # System message
    sys_msg = SystemMessage(
    content=f"You are a helpful assistant tasked with fetching relevant data for the user query. Book id: {state["book"]}")

    llm_response = llm_with_tools.invoke([sys_msg] + state["messages"])
    book = Book.objects.get(id=state["book"])
    Ticket.objects.create(query=state["messages"][0].content,
                          book=book,
                          response=llm_response.content)
    return {**state, "messages": [llm_response]}


def register_complaint(state: State) -> State:
    book, query = state["book"], state["messages"][0].content
    book = Book.objects.get(id=state["book"])
    Ticket.objects.create(query=query,
                          book=book,
                          response='Sorry about that, we have registered your complaint.')
    response = AIMessage(content='Sorry about that, we have registered your complaint.')
    return {**state, "messages": [response]}


def get_info(state: State) -> State:
    query = state["messages"][0].content
    response = similarity_search(query)
    book = Book.objects.get(id=state["book"])
    if response["distance"] >= 0.8:
        # Save a ticket for Human agent in database
        Ticket.objects.create(query=query,
                              book=book)
    else:
        response = response["context"]
        Ticket.objects.create(query=query,
                              book=book,
                              response=response)

    prompt = INFO_PROMPT

    prompt += f"""/n Query: {query}. Resonse: {response}"""
    llm_response = llm.invoke(prompt)
    return {
        **state,
        "messages": [AIMessage(content=llm_response.content)]
    }


def build_graph():
    from langgraph.prebuilt import tools_condition, ToolNode
    from langgraph.graph import START

    builder = StateGraph(State)

    builder.add_node("start_graph", start_graph)
    builder.add_node("get_info", get_info)

    builder.add_node("assistant", assistant)
    builder.add_node("register_complaint", register_complaint)
    builder.add_node("tools", ToolNode(tools))

    builder.add_edge(START, "start_graph")
    builder.add_conditional_edges(
        "start_graph",
        get_user_intent,
        {
            "info": "get_info",
            "query": "assistant",
            "complaint": "register_complaint"
        }
    )
    builder.add_conditional_edges(
        "assistant",
        # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
        # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
        tools_condition,
    )
    builder.add_edge("tools", "assistant")
    react_graph = builder.compile()

    return react_graph

In [43]:
graph = build_graph()

In [44]:
from langchain_core.messages import HumanMessage
s = {
    "book": 3,
    "messages": [HumanMessage(content="Pathetic service, i want to quit")]
}

In [45]:
result = graph.invoke(s)

In [46]:
result

{'messages': [HumanMessage(content='Pathetic service, i want to quit', additional_kwargs={}, response_metadata={}, id='9da1b6b0-59fc-49a0-b311-93a0a69738e6'),
  AIMessage(content="I'm here to help you. If you're experiencing issues or have concerns, please let me know what specific problems you're facing, and I'll do my best to assist you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 154, 'total_tokens': 188, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6c2953e649', 'id': 'chatcmpl-DjQc6tWxS90af4qjJZwJyJtMqiNoc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e5f8b-6d68-7b81-a5d7-03e3b48f912b-0', tool_calls=[], invalid_tool_cal